# Uplift modeling in marketing. Notebook 2: response-targeting baseline

Continuation of `01_Framing_EDA_EN.ipynb`. This notebook does not share the kernel of the previous one; it reloads deterministically the raw data and the arm and treatment constants. No EDA results are recalculated or repeated, as they already appear in notebook 1.

## Contents

- [Setup (resumed from S1 and S2)](#setup)
- [Section 3. Response-targeting baseline](#s3)
    - [3.1 Preparation of splits and pooled treatment](#s3-1)
    - [3.2 Response-targeting baseline $P(\text{visit}=1 \mid X)$ in the treated arm](#s3-2)
    - [3.3 Evaluation of the response-targeting baseline with uplift metrics in validation](#s3-3)
    - [3.4 Fast T-learner as provisional uplift reference](#s3-4)
    - [3.5 Agreement between propensity and uplift rankings](#s3-5)
    - [3.6 Preregistered confirmation rule for the sealed test](#s3-6)
    - [Synthesis of Section 3](#s3-summary)

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning

# Path bootstrap: allows `from src...` from the notebooks directory.
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import SEED
from src.i18n import make_lang
from src.viz import apply_plot_style

np.random.seed(SEED)
pd.set_option('display.max_columns', 50)
pd.set_option('display.precision', 4)
warnings.filterwarnings('ignore', category=FutureWarning)
# The internal causalml meta-learner propensity model (S4) uses an
# elastic-net solver that does not converge within the default max_iter on
# small samples. This does not affect the result because it is only a
# nuisance model, but it pollutes the output.
warnings.filterwarnings('ignore', category=ConvergenceWarning)

# Idioma canônico deste notebook é PT — passthrough, sem chamada de rede.
# EN edition: only this line is switched to make_lang('en').
lang = make_lang('en')

apply_plot_style()

In [ ]:
# Stack causal (verifique a instalação antes da primeira run)
# pip install econml causalml scikit-uplift shap mlflow

# Meta-learners e árvores de uplift
from causalml.inference.meta import BaseSRegressor, BaseTRegressor, BaseXRegressor, BaseRRegressor
from causalml.inference.tree import UpliftRandomForestClassifier

# Causal Forest com intervalos de confiança
from econml.dml import CausalForestDML
from econml.metalearners import TLearner, SLearner, XLearner

# Avaliação
from sklift.metrics import uplift_at_k, qini_auc_score, uplift_auc_score

from src.compat import patch_sklearn_matplotlib_support
patch_sklearn_matplotlib_support()
from sklift.viz import plot_qini_curve, plot_uplift_curve

# Interpretabilidade
import shap

In [ ]:
from src.config import ARTIFACTS_DIR, RETRAIN

ARTIFACTS_DIR.mkdir(exist_ok=True, parents=True)

# MLflow — ajuste o URI conforme seu setup Windows
# mlflow.set_tracking_uri('file:///C:/path/to/mlruns')
# mlflow.set_experiment('uplift_hillstrom')

<a id="setup"></a>

## Setup (resumed from S1 and S2)

The required inputs for Section 3 are reloaded here: raw data and the arm and treatment constants (`ARMS`, `TREATMENT`). This is the same deterministic reload used in `01_Framing_EDA_EN.ipynb`.

In [ ]:
from src.config import ARMS, TREATMENT_COL as TREATMENT
from src.data import load_hillstrom

df = load_hillstrom()

<a id="s3"></a>

# Section 3. Response-targeting baseline

This section establishes the benchmark for the project: a response model $P(\text{visit}=1 \mid X)$ trained solely on the treated arm, reproducing typical marketing ranking without explicit incremental-effect estimation. The validation set is then ranked by this probability and evaluated with the same uplift metrics used in the S6 confirmatory decision (Qini AUC and AUUC), enabling direct comparison against the S4 meta-learners and S5 models. That ranking is also compared via Spearman correlation against a fast, untuned T-learner under the working hypothesis of low agreement between response probability and incremental treatment effect. Finally, before any sealed-test metric exists, the S6 confirmation rule is recorded (Absolute Rule #2 of the protocol).

Treatment from S3 through S6 (pooled): $T=1$ denotes any e-mail arm (Mens or Womens) and $T=0$ denotes No E-Mail, totaling 42,694 treated observations against 21,306 controls. Individual arm separation returns only in S8 for the three-way assignment policy. No cell in this section touches the sealed test set (`src/splits.py::load_sealed_test`) or calls `predict` on it; only training and validation partitions are used.

<a id="s3-1"></a>

## 3.1 Preparation of splits and pooled treatment

Training and validation partitions are materialized via `make_splits`. Sections S3 through S6 use only these two partitions, keeping the sealed test stored on disk until S6. The pooled treatment (`treatment`: 1 for any e-mail, 0 for No E-Mail) is added via `add_pooled_treatment` without modifying the original `segment` column, which is reserved for the three-way policy in S8.

In [ ]:
from src.config import POOLED_TREATMENT_COL, PRIMARY_OUTCOME
from src.data import add_pooled_treatment
from src.splits import make_splits

df_pooled = add_pooled_treatment(df)
split_idx = make_splits(df_pooled)
train_df = df_pooled.loc[split_idx['train_idx']].copy()
val_df = df_pooled.loc[split_idx['val_idx']].copy()

labels = lang({
    'header': 'Tamanho das partições (teste permanece selado)',
    'training': 'Treino',
    'validation': 'Validação',
    'rows': 'linhas',
    'treated': 'tratados',
})
print(f"{labels['header']}:")
print(f"  {labels['training']}:   {len(train_df):>6} {labels['rows']} | {labels['treated']}: {int(train_df[POOLED_TREATMENT_COL].sum()):>6}")
print(f"  {labels['validation']}: {len(val_df):>6} {labels['rows']} | {labels['treated']}: {int(val_df[POOLED_TREATMENT_COL].sum()):>6}")

In [ ]:
from src.viz import plot_split_overview

labels = lang({
    'partition_row': 'Partição',
    'arm_row': 'Braço',
    'outcome_row': 'Outcome (visit)',
    'train': 'Treino',
    'val': 'Validação',
    'sealed': 'Teste selado (oculto até S6)',
    'outcome_neg': 'Não visitou (0)',
    'outcome_pos': 'Visitou (1)',
    'title': 'Composição do split',
    'subtitle': 'Treino e validação preservam a mesma proporção de braço e outcome; teste segue selado',
})
n_test = len(df_pooled) - len(train_df) - len(val_df)
fig, ax = plot_split_overview(
    train_df, val_df, n_test, TREATMENT, ARMS, PRIMARY_OUTCOME, labels,
    title=labels['title'], subtitle=labels['subtitle'],
)
plt.show()

Training and validation rows are ordered by (arm, outcome) only to display the relative proportions of each combination in blocks; the ordering carries no temporal meaning, as the dataset is not a time series. The grey sealed-test block shows partition size only, revealing no arm or outcome composition before Section 6. The visual similarity between training and validation blocks reflects the stratification (arm × visit) already verified by automated tests in `tests/test_suite.py`.

<a id="s3-2"></a>

## 3.2 Response-targeting baseline $P(\text{visit}=1 \mid X)$ in the treated arm

The baseline trains a response classifier solely on treated rows without seeing the control arm. It estimates who is most likely to visit the site among treated customers, without separating customers who would visit regardless from those who visit because of the e-mail.

LightGBM is used as the model class, matching the base learner shared by the four meta-learners in S4. Any performance difference relative to the meta-learners therefore stems from the targeting strategy (propensity versus uplift) rather than base-model capacity.

In [ ]:
from src.learners import fit_propensity_baseline, predict_propensity_score

propensity_model = fit_propensity_baseline(train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME)
val_df['propensity_score'] = predict_propensity_score(propensity_model, val_df)
val_df[['propensity_score']].describe()

<a id="s3-3"></a>

## 3.3 Evaluation of the response-targeting baseline with uplift metrics in validation

Although the baseline does not estimate incremental effects, its ranking is evaluated using the same metrics that govern S6: Qini AUC, uplift AUC (AUUC), and uplift@30%. These metrics require only a treatment-priority ranking, treating a high propensity score as high treatment priority.

In [ ]:
from src.evaluation import evaluate_ranking

baseline_metrics = evaluate_ranking(
    val_df[PRIMARY_OUTCOME].values, val_df['propensity_score'].values, val_df[POOLED_TREATMENT_COL].values,
)
print(baseline_metrics)

In [ ]:
from src.viz import add_chart_footer, add_chart_header

labels = lang({
    'title': 'Curva Qini — baseline de propensão (validação)',
    'series_name': 'Baseline de propensão',
    'subtitle': 'Baseline supera a linha aleatória, mas com margem modesta (Qini AUC = 0.0395)',
})
qini_disp = plot_qini_curve(
    val_df[PRIMARY_OUTCOME].values, val_df['propensity_score'].values, val_df[POOLED_TREATMENT_COL].values,
    perfect=True, name=labels['series_name'],
)
add_chart_header(qini_disp.figure_, title=labels['title'], subtitle=labels['subtitle'])
footer = lang({
    'text': 'Fonte: Validação, 12.800 linhas  |  Método: LightGBM sobre o braço tratado',
})['text']
add_chart_footer(qini_disp.figure_, text=footer)
qini_disp.figure_.subplots_adjust(top=0.78, bottom=0.15)
plt.show()

On validation (12,800 rows, 8,538 treated), the response-targeting baseline obtains Qini AUC = 0.0395, uplift AUC (AUUC) = 0.0235, and uplift@30% = +0.0923. Treating the top 30% of the propensity ranking yields a treated-versus-control visit-rate difference about 9.2pp higher than random assignment. These three values form the benchmark for the S4 meta-learners and S5 direct models; formal comparison with confidence intervals belongs to S6.

<a id="s3-4"></a>

## 3.4 Fast T-learner as provisional uplift reference

To check whether the propensity ranking agrees with an incremental-effect ranking, a simple T-learner (two independent LightGBM models, one per arm, without tuning) is fitted as a provisional CATE reference. The tuned, definitive version formally compared against the other meta-learners belongs to S4; this provisional model does not enter that comparison.

In [ ]:
from src.learners import fit_t_learner_quick, predict_t_learner_uplift

t_learner_models = fit_t_learner_quick(train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME)
val_df['t_learner_uplift'] = predict_t_learner_uplift(t_learner_models, val_df)
val_df[['t_learner_uplift']].describe()

<a id="s3-5"></a>

## 3.5 Agreement between propensity and uplift rankings

If the propensity ranking and the uplift ranking (T-learner) agreed closely, ranking by response probability would suffice for targeting. Spearman correlation between the two scores measures this overlap directly.

In [ ]:
from src.evaluation import spearman_ranking_correlation

rank_corr = spearman_ranking_correlation(val_df['propensity_score'], val_df['t_learner_uplift'])
print(rank_corr)

Spearman correlation between the propensity score and T-learner uplift is $\rho = 0.444$ ($p \approx 0$, expected with $n = 12,800$). This is higher than initially hypothesized: the rankings are correlated, indicating that customers with higher baseline visit probability also tend to respond more to the e-mail in this dataset. Even so, $\rho = 0.444$ leaves more than half of the ranking variation unshared between approaches, leaving room for uplift modeling to capture value missed by propensity modeling.

The untuned fast T-learner produced an estimated uplift with a long negative tail (minimum -0.58). This is not interpreted as evidence of Sleeping Dogs in this section, as the estimator is provisional and noisy; heterogeneity analysis is deferred to S7.

<a id="s3-6"></a>

## 3.6 Preregistered confirmation rule for the sealed test

Before any sealed-test metric exists, the rule that will determine in S6 whether the validation winner is confirmed on the sealed test is fixed.

> Confirmation rule for S6: the validation winner (highest Qini AUC across S3 through S5) is confirmed on the sealed test if and only if all three conditions hold: its sealed-test Qini is positive; the 95% bootstrap confidence interval of sealed-test Qini does not include zero; and its sealed-test Qini exceeds the response-targeting baseline Qini (Section 3.3) measured on the same test. If any condition fails, the outcome is reported as "not confirmed" without protocol adjustment (Absolute Rule #6: a negative result is a result).

This rule will not be revised in S6, regardless of what the validation results in S4 and S5 suggest.

<a id="s3-summary"></a>

### Synthesis of Section 3

| Item | Result |
|---|---|
| Response-targeting baseline: Qini AUC (validation) | 0.0395 |
| Response-targeting baseline: uplift@30% (validation) | +9.2pp |
| Correlation (Spearman) between propensity and fast T-learner | 0.444 ($p \approx 0$) |
| Confirmation rule for S6 | Defined in Section 3.6 before any test result |

The response-targeting baseline is established as the benchmark to beat. Moderate correlation with the T-learner confirms that propensity and uplift are related yet distinct signals, creating room to test whether the S4 meta-learners outperform this baseline on uplift metrics.

Next: [Section 4. Meta-learners (S/T/X/R)](03_Meta_Learners_EN.ipynb), using LightGBM as the shared base learner and testing the structural-advantage hypothesis for X-learner under imbalanced arms.